# Databricks notebook source

# # 10 · Validação da Gold
# **Fase 3 · Gold** | Concilia a Gold com a Silver, checa integridade e idempotência.
# Cada check usa `assert`: se algo falhar, a célula quebra e mostra qual.

In [0]:
CATALOG = "project_lakehouse_telemetria_ele"
SILVER = f"{CATALOG}.silver.telemetria_clean"
GOLD = f"{CATALOG}.gold"
C_ALARM = "alarme"
F_PICO, F_TENS, F_SENT = "temp_pico", "tensao_ausente", "corrente_sentinela"

ALARME_ATIVO = f"""(CASE WHEN {C_ALARM} = 'SOBRETEMP' THEN 1 ELSE 0 END)"""   # valores reais: OK | SOBRETEMP
def fi(c): return f"COALESCE(CAST({c} AS INT),0)"

def one(sql):
    return spark.sql(sql).first()

# Esperados (Fase 2)
ESP = {"leituras": 1_050_000, "picos": 3_150, "tensao_ausente": 10_500, "corrente_sentinela": 2_100}

In [0]:
## 1. Conciliação com a Silver

silver = one(f"""SELECT COUNT(*) AS leituras, SUM({fi(F_PICO)}) AS picos, SUM({ALARME_ATIVO}) AS alarmes,
  SUM({fi(F_TENS)}) AS tensao_ausente, SUM({fi(F_SENT)}) AS corrente_sentinela FROM {SILVER}""").asDict()

def somas(tabela):
    return one(f"""SELECT SUM(leituras) AS leituras, SUM(qtd_picos) AS picos, SUM(qtd_alarmes) AS alarmes,
      SUM(qtd_tensao_ausente) AS tensao_ausente, SUM(qtd_corrente_sentinela) AS corrente_sentinela
      FROM {GOLD}.{tabela}""").asDict()

rows = []
for tab in ["dispositivo_hora", "dispositivo_dia"]:
    g = somas(tab)
    for k in silver:
        ok = int(g[k]) == int(silver[k])
        rows.append((tab, k, int(silver[k]), int(g[k]), "OK" if ok else "FALHA"))
        assert ok, f"{tab}.{k}: Silver={silver[k]} Gold={g[k]}"
for k, v in ESP.items():
    assert int(silver[k]) == v, f"Silver {k}={silver[k]} difere do esperado {v}"

display(spark.createDataFrame(rows, "tabela string, metrica string, silver long, gold long, status string"))

In [0]:
## 2. Grão e chaves

h = one(f"""SELECT COUNT(*) AS linhas, COUNT(DISTINCT device_id, hora_ts) AS chaves,
            COUNT(DISTINCT device_id) AS dispositivos FROM {GOLD}.dispositivo_hora""")
assert h.linhas == h.chaves, "Chave repetida em dispositivo_hora"
assert h.linhas <= 8400, f"dispositivo_hora tem {h.linhas} linhas (máx. 50 x 168 = 8400)"
assert h.dispositivos == 50

d = one(f"""SELECT COUNT(*) AS linhas, COUNT(DISTINCT device_id, data_evento) AS chaves
            FROM {GOLD}.dispositivo_dia""")
assert d.linhas == d.chaves, "Chave repetida em dispositivo_dia"

p = one(f"SELECT COUNT(*) AS linhas, COUNT(DISTINCT hora_do_dia) AS horas FROM {GOLD}.perfil_horario")
assert p.linhas == 24 and p.horas == 24

print("dispositivo_hora:", h.linhas, "| dispositivo_dia:", d.linhas, "| perfil_horario:", p.linhas)

In [0]:
## 3. Ranking

from statistics import NormalDist
Z_LIMITE = round(NormalDist().inv_cdf(1 - 0.05 / (3 * 50)), 3)   # mesmo critério do notebook 09

r = one(f"""SELECT COUNT(*) AS linhas, COUNT(DISTINCT device_id) AS ids, COUNT(DISTINCT posicao) AS posicoes,
            MIN(posicao) AS pmin, MAX(posicao) AS pmax,
            SUM(CASE WHEN z_max IS NULL OR isnan(z_max) THEN 1 ELSE 0 END) AS z_invalidos,
            SUM(CASE WHEN (z_max >= {Z_LIMITE}) <> (nivel = 'atenção') THEN 1 ELSE 0 END) AS nivel_incoerente,
            SUM(CASE WHEN nivel = 'atenção' THEN 1 ELSE 0 END) AS em_atencao,
            MAX(z_max) AS z_max_obs FROM {GOLD}.ranking_risco_ativos""")
assert r.linhas == 50 and r.ids == 50, "Ranking deve ter 50 ativos"
assert r.posicoes == 50 and r.pmin == 1 and r.pmax == 50, "Posições devem ser únicas de 1 a 50"
assert r.z_invalidos == 0, "z_max nulo ou NaN"
assert r.nivel_incoerente == 0, "Nível inconsistente com o limite"
print(f"ranking OK | limite z = {Z_LIMITE} | z máximo observado = {r.z_max_obs} | ativos em atenção = {r.em_atencao}")


In [0]:
## 4. Sanidade estatística (informativa, não falha o notebook)
# Sob ruído puro o desvio dos z fica perto de 1. Desvio bem acima de 1 indica diferença real entre ativos.

display(spark.sql(f"""
SELECT ROUND(STDDEV(z_picos),3) AS desvio_z_picos, ROUND(STDDEV(z_tensao),3) AS desvio_z_tensao,
       ROUND(STDDEV(z_sentinela),3) AS desvio_z_sentinela
FROM {GOLD}.ranking_risco_ativos
"""))

In [0]:
## 5. Idempotência: reexecutar 08 e 09 e comparar contagens

def snapshot():
    return {
        "dispositivo_hora": spark.table(f"{GOLD}.dispositivo_hora").count(),
        "dispositivo_dia": spark.table(f"{GOLD}.dispositivo_dia").count(),
        "perfil_horario": spark.table(f"{GOLD}.perfil_horario").count(),
        "ranking_risco_ativos": spark.table(f"{GOLD}.ranking_risco_ativos").count(),
        "soma_leituras_hora": int(somas("dispositivo_hora")["leituras"]),
    }

antes = snapshot()
dbutils.notebook.run("./08_gold_agregacoes", 1800)
dbutils.notebook.run("./09_gold_ranking_risco", 1800)
depois = snapshot()

print("antes :", antes)
print("depois:", depois)
assert antes == depois, "Reexecução alterou as contagens"
print("Idempotência OK")